# 03. Policy & Modules

ArcAgent has **two** governance layers that wrap every tool dispatch:

1. **Policy pipeline** (from `arctrust.policy`) — a deterministic,
   first-DENY-wins evaluator that authorizes whether a tool may be
   called at all. Federal-grade. Fail-closed.
2. **Module Bus** (`arcagent.core.module_bus`) — an asynchronous
   event bus where in-process *extensions* subscribe to lifecycle
   events (`agent:pre_tool`, `agent:post_tool`, `agent:ready`, …)
   and can observe, audit, transform, or **veto** behaviour.

Policy is the *gate*. Bus subscribers are the *extension surface*.
They compose: policy runs first, then `agent:pre_tool` fires, and any
subscribed handler can veto; either path raises a structured error
out of the registry.

**You will learn:**

- How `ArcAgent` wires `arctrust.policy.PolicyPipeline` at the
  tool-dispatch boundary (cross-ref `arctrust/03-policy-pipeline`
  for the deeper policy walkthrough).
- How a policy DENY surfaces — `PolicyDenied` from the engine,
  `ToolVetoedError` from the bus.
- The `ModuleBus`, `ModuleContext`, and `EventContext` primitives —
  and the `subscribe`/`emit` surface (the bus has no module registry;
  it is purely a subscribe/dispatch fabric).
- How to write a tiny extension that subscribes handlers to the bus.
- `ModuleBusError` and the failure modes it represents.
- The end-to-end call flow when policy + multiple subscribers + audit
  are all stacked on a single tool call.

All cells run with **no API key**. Crypto is real (cheap); LLM
calls are not used in this notebook.

## 1. Setup

The boilerplate adds every package's `src/` (and `tests/`, where
present) to `sys.path`. After this cell, `import arcagent`,
`import arctrust`, and `import arcrun` all resolve from the
in-repo packages.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Pull in the pieces we'll exercise. `arcagent` re-exports the
error hierarchy (`ToolVetoedError`, `ModuleBusError`); the bus
primitives live under `arcagent.core.module_bus`; and the policy
engine is re-exported through `arcagent.core.tool_policy` (which
is itself a thin shim over `arctrust.policy`).

In [2]:
import asyncio
from typing import Any

import arcagent
from arcagent import ModuleBusError, ToolVetoedError
from arcagent.core.module_bus import (
    EventContext,
    ModuleBus,
    ModuleContext,
)
from arcagent.core.tool_policy import (
    PolicyContext,
    PolicyDenied,
    PolicyPipeline,
    ToolCall,
    build_pipeline,
)

print("arcagent", arcagent.__version__)
print("ModuleBus surface:", [m for m in dir(ModuleBus) if not m.startswith("_")])
print("ToolVetoedError MRO root:", ToolVetoedError.__mro__[1].__name__)

arcagent 0.15.0
ModuleBus surface: ['emit', 'handler_count', 'handler_count_by_module', 'subscribe']
ToolVetoedError MRO root: ToolError


## 2. Two layers of governance

Every tool dispatch in ArcAgent goes through this exact
sequence (see `ToolRegistry._create_wrapped_execute`):

```
0. Argument schema validation
1. PolicyPipeline.evaluate()         ← arctrust.policy
   └─ DENY → raise PolicyDenied      (subclass of ArcAgentError)
2. ModuleBus.emit('agent:pre_tool')  ← arcagent.core.module_bus
   └─ ctx.veto(reason)               → raise ToolVetoedError
3. tool.execute(**args)              under timeout + telemetry span
4. ModuleBus.emit('agent:post_tool')
5. AgentTelemetry.audit_event(...)
```

Two distinct authorities, two distinct exceptions. The policy
pipeline is the **synchronous, deterministic gate** — same call,
same context, same answer, every time. The module bus is the
**async, pluggable extension surface** — handlers are arbitrary
Python coroutines and may consult mutable state.

Why both? Different design goals:

| Concern | Policy pipeline | Module bus |
|---|---|---|
| Decision is | Authoritative authorization | Observation + opt-in veto |
| Determinism | Required | Not required |
| Fail mode | Closed (deny) | Closed (veto) |
| Use cases | Tier rules, allowlists, supply-chain checks | Audit hooks, HITL prompts, telemetry shims |
| Lives in | `arctrust.policy` | `arcagent.core.module_bus` |


Quick visual check that both surfaces exist as advertised. We
haven't wired them together yet — that's section 4 onwards.

In [3]:
bus = ModuleBus()
pipeline = build_pipeline(tier="personal")

print("ModuleBus instance     :", type(bus).__name__)
print("Default handler counts :", bus.handler_count("agent:pre_tool"))
print("PolicyPipeline (personal):", type(pipeline).__name__)
print("Layer count             :", len(pipeline._layers))
print("Layer names             :", [layer.name for layer in pipeline._layers])

ModuleBus instance     : ModuleBus
Default handler counts : 0
PolicyPipeline (personal): PolicyPipeline
Layer count             : 2
Layer names             : ['identity', 'global']


## 3. PolicyPipeline integration

`ArcAgent.startup()` builds a `PolicyPipeline` from the agent's
tier and `arcagent.toml`, then hands it to `ToolRegistry`:

```python
self._tool_registry = ToolRegistry(
    config=self._config.tools,
    bus=self._bus,
    telemetry=self._telemetry,
    policy_pipeline=pipeline,        # arctrust.policy.PolicyPipeline
    agent_did=self._identity.did,    # set by arctrust.identity
    tier=self._config.tier,
)
```

From there, every wrapped execute path calls `pipeline.evaluate`
before the bus ever emits `agent:pre_tool`. A DENY raises
`PolicyDenied`, which carries the structured `Decision` (layer,
rule_id, reason) for the audit trail.

For a deep dive on the engine itself — layer composition,
shadow mode, restricted-mode safe sets, tier matrix — see
[`arctrust/03-policy-pipeline.ipynb`](../arctrust/03-policy-pipeline.ipynb).
Here we focus on the *integration point*: what the agent does
with the engine.

Build the smallest possible deny-rule pipeline: any call to
`bash` is rejected by the global layer.

In [4]:
from arctrust.identity import AgentIdentity
from arctrust.policy import sign_call

# Every ToolCall must be signed. The IdentityLayer runs first at EVERY tier
# and denies unsigned/forged calls fail-closed (the "SSH-key model": nothing
# runs unless the call is signed by the agent that holds the private key for
# the DID it claims). We mint a self-signed identity; personal tier admits any
# validly self-signed key. ``sign_call`` rebinds the call to the signer and
# attaches the public key + signature.
IDENTITY = AgentIdentity.generate(org="demo", agent_type="operator")
AGENT_DID = IDENTITY.did
print("agent DID:", AGENT_DID)

deny_pipeline = build_pipeline(
    tier="personal",
    global_deny_rules={"bash": "bash is not allowed in this notebook"},
)

ctx = PolicyContext(
    tier="personal",
    policy_version="v1",
    bundle_age_seconds=0.0,
)

deny_call = sign_call(
    ToolCall(
        tool_name="bash",
        arguments={"cmd": "ls"},
        agent_did=AGENT_DID,
        session_id="sess_demo",
        classification="unclassified",
    ),
    IDENTITY,
)

decision = await deny_pipeline.evaluate(deny_call, ctx)
print("outcome :", decision.outcome)
print("layer   :", decision.layer)
print("rule_id :", decision.rule_id)
print("reason  :", decision.reason)

agent DID: did:arc:demo:operator/be4125d2
outcome : deny
layer   : global
rule_id : global.denylist
reason  : bash is not allowed in this notebook


An ALLOW for the same pipeline shape — different tool name,
no rule matches, the global layer falls through.

In [5]:
allow_call = sign_call(
    ToolCall(
        tool_name="read_file",
        arguments={"path": "notes/agenda.md"},
        agent_did=AGENT_DID,
        session_id="sess_demo",
        classification="unclassified",
    ),
    IDENTITY,
)

decision = await deny_pipeline.evaluate(allow_call, ctx)
print("outcome :", decision.outcome)
print("layer   :", decision.layer or "(none)")
print("reason  :", decision.reason or "(none)")

outcome : allow
layer   : (none)
reason  : (none)


## 4. Surfacing a veto: PolicyDenied vs ToolVetoedError

Two errors, two surfaces, two reasons. To keep the notebook
focused on the integration boundary, we'll wire a tiny
*dispatcher* that mimics `ToolRegistry._create_wrapped_execute`
exactly — policy first, bus second, execute third — without
spinning up a full `ArcAgent`. This is the same control flow
the real registry uses; see `tool_registry.py` lines 518–600.

In [6]:
async def dispatch(
    *,
    pipeline: PolicyPipeline,
    bus: ModuleBus,
    tool_name: str,
    arguments: dict[str, Any],
    execute,
    agent_did: str = AGENT_DID,
) -> Any:
    """Mirror of ToolRegistry._create_wrapped_execute (cut down)."""
    # 1. Policy gate — the real registry signs every ToolCall with the agent's
    # key before evaluating; we do the same via sign_call so the IdentityLayer
    # authenticates it.
    call = sign_call(
        ToolCall(
            tool_name=tool_name,
            arguments=arguments,
            agent_did=agent_did,
            session_id="sess_demo",
            classification="unclassified",
        ),
        IDENTITY,
    )
    pol_ctx = PolicyContext(tier="personal", policy_version="v1", bundle_age_seconds=0.0)
    decision = await pipeline.evaluate(call, pol_ctx)
    if decision.is_deny():
        raise PolicyDenied(decision)

    # 2. Pre-tool event (may veto)
    ev = await bus.emit(
        "agent:pre_tool",
        {"tool": tool_name, "args": arguments},
        agent_did=agent_did,
    )
    if ev.is_vetoed:
        raise ToolVetoedError(
            message=f"Tool '{tool_name}' vetoed: {ev.veto_reason}",
            details={"tool": tool_name, "reason": ev.veto_reason},
        )

    # 3. Execute
    result = await execute(**arguments)

    # 4. Post-tool event
    await bus.emit("agent:post_tool", {"tool": tool_name, "result": result})
    return result


async def echo(text: str = "") -> str:
    return f"echo: {text}"

Happy path first — fresh `ModuleBus`, empty pipeline, allowed
tool. The dispatcher returns the executed value cleanly.

In [7]:
bus = ModuleBus()
open_pipeline = build_pipeline(tier="personal")  # no deny rules

result = await dispatch(
    pipeline=open_pipeline,
    bus=bus,
    tool_name="echo",
    arguments={"text": "hello"},
    execute=echo,
)
print("result:", result)

result: echo: hello


Now register a global deny rule for `bash`. Because the policy
pipeline runs *first*, the bus never sees the call — handlers
subscribed to `agent:pre_tool` would not fire on this attempt.

In [8]:
denied_pipeline = build_pipeline(
    tier="personal",
    global_deny_rules={"bash": "bash blocked at notebook tier"},
)

try:
    await dispatch(
        pipeline=denied_pipeline,
        bus=bus,
        tool_name="bash",
        arguments={"cmd": "ls"},
        execute=echo,
    )
except PolicyDenied as exc:
    print("policy denied:", exc)
    print("layer       :", exc.decision.layer)
    print("rule_id     :", exc.decision.rule_id)
    print("code        :", exc.code)

policy denied: [POLICY_DENIED] tool_policy: [global:global.denylist] bash blocked at notebook tier
layer       : global
rule_id     : global.denylist
code        : POLICY_DENIED


And the *other* veto path — policy passes, but a subscribed
handler stops the call by calling `ctx.veto(...)`. This is the
shape that `ToolVetoedError` exists for: a human-in-the-loop
module, a runtime resource check, or any pluggable extension
that needs to stop a single call without rewriting policy.

In [9]:
bus2 = ModuleBus()


async def block_destructive(ctx: EventContext) -> None:
    if ctx.data.get("args", {}).get("text", "").startswith("rm "):
        ctx.veto("refusing destructive echo")


bus2.subscribe("agent:pre_tool", block_destructive, priority=10)

try:
    await dispatch(
        pipeline=open_pipeline,
        bus=bus2,
        tool_name="echo",
        arguments={"text": "rm -rf /"},
        execute=echo,
    )
except ToolVetoedError as exc:
    print("tool vetoed:", exc.message)
    print("code       :", exc.code)
    print("details    :", exc.details)

tool vetoed: Tool 'echo' vetoed: refusing destructive echo
code       : TOOL_VETOED
details    : {'tool': 'echo', 'reason': 'refusing destructive echo'}


**Takeaway.** `PolicyDenied` and `ToolVetoedError` are not
interchangeable. They tell auditors which authority refused the
call. Both inherit from `ArcAgentError`, both carry structured
context, and both surface from the same dispatch boundary.

## 5. ModuleBus basics

The bus has exactly two surfaces:

1. **`subscribe(event, handler, priority=100, module_name="", timeout_seconds=30)`**
   — register an async handler for an event name. Lower priority
   runs first; same-priority handlers run concurrently via
   `asyncio.gather`. Default priority is `100`. Default timeout
   is 30 s; a slow handler is logged-and-dropped, not raised.
2. **`emit(event, data, agent_did="", trace_id="") -> EventContext`**
   — fire an event. Returns the `EventContext` so callers can
   inspect veto state.

That's it — there is **no** module registry, no `register_module`,
no `startup`/`shutdown` on the bus. The bus is a pure subscribe/dispatch
fabric. An *extension* is simply any object (or module-level function)
that subscribes handlers to the bus. In production those handlers come
from `@hook(event=...)`-decorated functions the `CapabilityLoader`
discovers on disk and bridges onto the bus via `subscribe(...)`; in this
notebook we call `subscribe(...)` directly.

Bus handlers are *fire-and-forget* with respect to exceptions:
any exception raised inside a handler is logged, then swallowed.
Failure isolation is by design — one buggy handler cannot crash
another. (This is what `ModuleBusError` is *not* — see §8.)

Smallest possible bus interaction. Note that handlers are
coroutines — sync handlers will fail-await.

In [10]:
events: list[str] = []


async def trace_handler(ctx: EventContext) -> None:
    events.append(f"{ctx.event}:{ctx.data}")


small_bus = ModuleBus()
small_bus.subscribe("greeting", trace_handler)
await small_bus.emit("greeting", {"name": "ada"})
await small_bus.emit("greeting", {"name": "alan"})
print("captured events:", events)

captured events: ["greeting:{'name': 'ada'}", "greeting:{'name': 'alan'}"]


Handler **priority** controls ordering. By convention:
`10`=policy-shaped, `50`=security, `100`=default, `200`=logging.
Within a priority group, order is undefined; across groups,
lower runs first.

In [11]:
order: list[int] = []


async def at(p: int):
    async def h(ctx: EventContext) -> None:
        order.append(p)

    return h


ord_bus = ModuleBus()
ord_bus.subscribe("ev", await at(200), priority=200)
ord_bus.subscribe("ev", await at(10), priority=10)
ord_bus.subscribe("ev", await at(100), priority=100)

await ord_bus.emit("ev", {})
print("priority order :", order)
print("handler_count  :", ord_bus.handler_count("ev"))

priority order : [10, 100, 200]
handler_count  : 3


`ModuleContext` is the dependency-injection record handed to a
`@capability` class's `setup(ctx)` method (the lifecycle contract the
`CapabilityLoader` drives — see notebook 04 §3). It is a frozen
dataclass — a capability cannot reassign `bus`, `tool_registry`, etc.,
but the objects themselves remain mutable, so a capability can call
`tool_registry.register(...)` to publish new tools or `bus.subscribe(...)`
to add handlers.

We won't construct a real one in this notebook — the fields
are heavyweight (`AgentTelemetry`, full `ArcAgentConfig`). The
shape:

In [12]:
from dataclasses import fields

for f in fields(ModuleContext):
    print(f"{f.name:>14} : {f.type}")

           bus : ModuleBus
 tool_registry : ToolRegistry
        config : ArcAgentConfig
     telemetry : AgentTelemetry
     workspace : Path
    llm_config : LLMConfig


## 6. A trivial custom module

There is no `Module` base class or Protocol — an extension is just an
object that subscribes handlers to the bus. The convention the
`CapabilityLoader`'s hook bridge uses is:

```python
bus.subscribe(event, handler, priority=..., module_name=...)
```

Each subscription is tagged with a `module_name` so the bus can find
and de-duplicate an extension's handlers across reloads
(`handler_count_by_module`).

Below is the smallest meaningful extension: an in-memory event recorder.
Its `install(bus)` method subscribes to `agent:pre_tool` and
`agent:post_tool` and stores a copy of every event for later inspection.
On disk this same recorder would be two `@hook`-decorated functions in a
`capabilities.py` file — the loader would call `bus.subscribe(...)` for
you.

In [13]:
class EventRecorder:
    """Captures pre/post tool events for inspection.

    An extension is just an object that subscribes handlers to the bus.
    ``install(bus)`` mirrors what the CapabilityLoader's hook bridge does
    for ``@hook``-decorated functions: ``bus.subscribe(event, handler,
    module_name=...)``.
    """

    NAME = "event_recorder"

    def __init__(self) -> None:
        self.events: list[tuple[str, dict[str, Any]]] = []

    def install(self, bus: ModuleBus) -> None:
        bus.subscribe("agent:pre_tool", self._capture, module_name=self.NAME)
        bus.subscribe("agent:post_tool", self._capture, module_name=self.NAME)

    async def _capture(self, ctx: EventContext) -> None:
        self.events.append((ctx.event, dict(ctx.data)))


recorder_probe = EventRecorder()
print("EventRecorder public methods:", [m for m in dir(recorder_probe) if not m.startswith("_")])

EventRecorder public methods: ['NAME', 'events', 'install']


Wire it up. Because the extension talks to the bus directly, there is no
`ModuleContext` to construct and no `register_module`/`startup` step — we
just hand it the bus and it subscribes. `handler_count_by_module` confirms
the two subscriptions landed under the extension's `module_name`.

In [14]:
rec_bus = ModuleBus()
recorder = EventRecorder()
recorder.install(rec_bus)

print("pre handlers :", rec_bus.handler_count("agent:pre_tool"))
print("post handlers:", rec_bus.handler_count("agent:post_tool"))
print(
    "handlers for this extension:",
    rec_bus.handler_count_by_module("agent:pre_tool", EventRecorder.NAME),
)

pre handlers : 1
post handlers: 1
handlers for this extension: 1


Drive a tiny synthetic interaction through `dispatch` and let
the recorder capture both the pre and post events.

In [15]:
result = await dispatch(
    pipeline=open_pipeline,
    bus=rec_bus,
    tool_name="echo",
    arguments={"text": "hi from a module"},
    execute=echo,
)

print("dispatch result :", result)
print("captured events :")
for ev_name, ev_data in recorder.events:
    print(f"  - {ev_name:>16} {ev_data}")

dispatch result : echo: hi from a module
captured events :
  -   agent:pre_tool {'tool': 'echo', 'args': {'text': 'hi from a module'}}
  -  agent:post_tool {'tool': 'echo', 'result': 'echo: hi from a module'}


There is no bus-level teardown to call — handlers live on the bus until
the bus itself is discarded. (Resource-owning extensions that need
explicit acquire/release use the `@capability` lifecycle contract instead;
see notebook 04 §3.) The recorder simply keeps capturing until we stop
emitting.

In [16]:
# Handlers persist on the bus; there is no bus.shutdown().
# Confirm the recorder is still subscribed and holding its captures.
print("still subscribed:", rec_bus.handler_count_by_module("agent:pre_tool", EventRecorder.NAME))
print("events captured :", len(recorder.events))

still subscribed: 1
events captured : 2


## 7. EventContext fields

Every handler receives a single `EventContext` argument. The
shape:

| Field | Meaning |
|---|---|
| `event` | Event name string, e.g. `"agent:pre_tool"` |
| `data` | Snapshot of the dict passed to `bus.emit(...)` |
| `agent_did` | DID of the agent emitting the event |
| `trace_id` | Correlation id for an end-to-end run |
| `is_vetoed` | True if any handler called `ctx.veto(reason)` |
| `veto_reason` | First veto reason (later vetoes do not overwrite) |

Two important guarantees:

- **Snapshot copy.** `__post_init__` copies `data` so a handler
  cannot mutate the original dict the caller passed to `emit`.
- **First veto wins.** A second `ctx.veto(...)` is a no-op; the
  reason from the first caller is preserved for the audit
  trail.

Confirm the snapshot semantics — we mutate the dict the caller
originally constructed and the `EventContext.data` we observe
from inside the handler is unchanged.

In [17]:
snap_bus = ModuleBus()
captured: dict[str, Any] = {}


async def grab(ctx: EventContext) -> None:
    captured.update(ctx.data)


snap_bus.subscribe("test", grab)

payload = {"tool": "read_file", "args": {"path": "/etc/passwd"}}
await snap_bus.emit("test", payload)

# Mutate the original payload AFTER emit — should not leak.
payload["tool"] = "rm"
print("original payload :", payload)
print("captured snapshot:", captured)

original payload : {'tool': 'rm', 'args': {'path': '/etc/passwd'}}
captured snapshot: {'tool': 'read_file', 'args': {'path': '/etc/passwd'}}


Confirm first-veto-wins. The audit trail must reflect *who*
vetoed first, not the last subscriber to chime in.

In [18]:
ctx = EventContext(event="agent:pre_tool", data={}, agent_did=AGENT_DID, trace_id="t1")
ctx.veto("first reason: policy violation")
ctx.veto("second reason: rate limit")
print("vetoed     :", ctx.is_vetoed)
print("veto reason:", ctx.veto_reason)

vetoed     : True
veto reason: first reason: policy violation


## 8. ModuleBusError

`ModuleBusError` is the structured exception type for **bus
lifecycle failures**: a module that fails to start, a handler
wired in a way the bus refuses to accept, an explicit error
raised by infrastructure that wraps the bus.

It is **not** raised by:

- A handler crashing — those are caught and logged inside
  `ModuleBus._run_handler`. Failure isolation is the whole
  point.
- A handler timing out — same path; logged at WARNING, the
  remaining handlers in the priority group still run.
- A veto — that surfaces as `ToolVetoedError` from the
  registry, *not* `ModuleBusError`.

Like every `ArcAgentError`, it carries a machine-readable
`code`, the originating `component`, and a `details` dict for
audit consumers.

First, the trivial shape — what callers see when they catch one.

In [19]:
err = ModuleBusError(
    code="MODULE_STARTUP_FAILED",
    message="Module 'recorder' failed to start: telemetry sink unreachable",
    details={"module": "recorder", "phase": "startup"},
)
print("str(err)  :", err)
print("code      :", err.code)
print("component :", err.component)
print("details   :", err.details)

str(err)  : [MODULE_STARTUP_FAILED] module_bus: Module 'recorder' failed to start: telemetry sink unreachable
code      : MODULE_STARTUP_FAILED
component : module_bus
details   : {'module': 'recorder', 'phase': 'startup'}


Confirm that handler exceptions are **isolated**, not surfaced
as `ModuleBusError`. Two handlers fire for the same event; the
second one raises; the first still runs cleanly; `emit` does
not raise.

In [20]:
ran: list[str] = []


async def good(ctx: EventContext) -> None:
    ran.append("good")


async def bad(ctx: EventContext) -> None:
    ran.append("bad-before-raise")
    raise RuntimeError("simulated handler failure")


iso_bus = ModuleBus()
iso_bus.subscribe("ev", good)
iso_bus.subscribe("ev", bad)

# emit() does not raise — exceptions are logged and swallowed.
result = await iso_bus.emit("ev", {})
print("ran handlers :", ran)
print("emit returned:", type(result).__name__)
print("vetoed       :", result.is_vetoed)

Handler bad for event ev raised an exception
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-af67a1128a112c184/packages/arcagent/src/arcagent/core/module_bus.py", line 167, in _run_handler
    await asyncio.wait_for(
  File "/Users/joshschultz/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/asyncio/tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
  File "/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_17978/1424894236.py", line 10, in bad
    raise RuntimeError("simulated handler failure")
RuntimeError: simulated handler failure


ran handlers : ['good', 'bad-before-raise']
emit returned: EventContext
vetoed       : False


Confirm timeout isolation. We register a slow handler with a
100 ms timeout; `emit` returns without raising and the second
handler still runs.

In [21]:
import asyncio

to_ran: list[str] = []


async def slow(ctx: EventContext) -> None:
    await asyncio.sleep(1.0)
    to_ran.append("slow-finished-which-it-shouldnt")


async def fast(ctx: EventContext) -> None:
    to_ran.append("fast")


to_bus = ModuleBus()
to_bus.subscribe("ev", slow, timeout_seconds=0.05)
to_bus.subscribe("ev", fast)

import time as _t

t0 = _t.monotonic()
await to_bus.emit("ev", {})
elapsed_ms = (_t.monotonic() - t0) * 1000
print(f"emit elapsed: {elapsed_ms:.0f} ms (slow handler timed out)")
print("ran         :", to_ran)

Handler slow for event ev timed out after 0.1s


emit elapsed: 52 ms (slow handler timed out)
ran         : ['fast']


## 9. Stacking — policy + modules + audit, end-to-end

When a real `ArcAgent` runs a tool, the call passes through
every layer in this order. Same control flow as the dispatcher
we built in §4 — just with the full registry, telemetry, and
audit emission wired in.

```
LLM      Loop       ToolRegistry          Pipeline       ModuleBus           Tool        Audit
  |        |              |                   |              |                |            |
  | tool_call             |                   |              |                |            |
  |------->|              |                   |              |                |            |
  |        | dispatch     |                   |              |                |            |
  |        |------------->|                   |              |                |            |
  |        |              | validate args     |              |                |            |
  |        |              | evaluate          |              |                |            |
  |        |              |------------------>|              |                |            |
  |        |              |                   | layers...    |                |            |
  |        |              |     allow         |              |                |            |
  |        |              |<------------------|              |                |            |
  |        |              | emit pre_tool                    |                |            |
  |        |              |--------------------------------->|                |            |
  |        |              |                                  | recorder hook  |            |
  |        |              |                                  | gate hook      |            |
  |        |              |     EventContext (no veto)       |                |            |
  |        |              |<---------------------------------|                |            |
  |        |              | execute(args)                                     |            |
  |        |              |-------------------------------------------------->|            |
  |        |              |                                                  | result      |
  |        |              |<--------------------------------------------------|            |
  |        |              | emit post_tool                                                 |
  |        |              |--------------------------------->|                             |
  |        |              | audit_event                                                    |
  |        |              |--------------------------------------------------------------->|
  |        | result       |                                                                |
  |        |<-------------|                                                                |
  | result |              |                                                                |
  |<-------|              |                                                                |
```

Failure paths short-circuit:

- DENY at any layer → `PolicyDenied` thrown before the bus
  emits anything.
- Bus veto on `agent:pre_tool` → `ToolVetoedError` thrown
  before `tool.execute(...)`.
- Tool timeout → `ToolError(code="TOOL_TIMEOUT")` thrown after
  the policy + bus gates have already passed.

Either way, the audit trail records the boundary that refused.
Single emission point, structured details, fail-closed.

Concrete stack: open pipeline + recorder module + a tiny
rate-limit-style gate module. We'll demonstrate (a) success,
(b) policy DENY, (c) bus veto, all on the same dispatcher.

In [22]:
class CallCounter:
    """Veto after N calls — synthetic rate limiter."""

    NAME = "call_counter"

    def __init__(self, *, limit: int) -> None:
        self.count = 0
        self._limit = limit

    def install(self, bus: ModuleBus) -> None:
        bus.subscribe("agent:pre_tool", self._gate, priority=10, module_name=self.NAME)

    async def _gate(self, ctx: EventContext) -> None:
        self.count += 1
        if self.count > self._limit:
            ctx.veto(f"call limit {self._limit} exceeded")

Build the stacked rig.

In [23]:
stack_bus = ModuleBus()
recorder = EventRecorder()
limiter = CallCounter(limit=2)
recorder.install(stack_bus)
limiter.install(stack_bus)

stacked_pipeline = build_pipeline(
    tier="personal",
    global_deny_rules={"bash": "denied at notebook tier"},
)

print("extensions installed:", [EventRecorder.NAME, CallCounter.NAME])
print("pre_tool handlers   :", stack_bus.handler_count("agent:pre_tool"))
print("post_tool handlers  :", stack_bus.handler_count("agent:post_tool"))

extensions installed: ['event_recorder', 'call_counter']
pre_tool handlers   : 2
post_tool handlers  : 1


Two successful dispatches inside the rate limit. Recorder sees
both pre and post events for each call.

In [24]:
for i in range(2):
    await dispatch(
        pipeline=stacked_pipeline,
        bus=stack_bus,
        tool_name="echo",
        arguments={"text": f"call {i}"},
        execute=echo,
    )

print("calls counted        :", limiter.count)
print("recorder event count :", len(recorder.events))
for ev_name, ev_data in recorder.events:
    print(f"  - {ev_name:>16} {ev_data}")

calls counted        : 2
recorder event count : 4
  -   agent:pre_tool {'tool': 'echo', 'args': {'text': 'call 0'}}
  -  agent:post_tool {'tool': 'echo', 'result': 'echo: call 0'}
  -   agent:pre_tool {'tool': 'echo', 'args': {'text': 'call 1'}}
  -  agent:post_tool {'tool': 'echo', 'result': 'echo: call 1'}


Third call is over the limit — `CallCounter._gate` vetoes,
`dispatch` raises `ToolVetoedError`. Recorder still captured
the `agent:pre_tool` event (handlers all run even on veto).

In [25]:
try:
    await dispatch(
        pipeline=stacked_pipeline,
        bus=stack_bus,
        tool_name="echo",
        arguments={"text": "third"},
        execute=echo,
    )
except ToolVetoedError as exc:
    print("vetoed:", exc.message)

print("calls counted        :", limiter.count)
print("recorder event count :", len(recorder.events))

vetoed: Tool 'echo' vetoed: call limit 2 exceeded
calls counted        : 3
recorder event count : 5


And policy DENY short-circuits ahead of the bus entirely —
the limiter's `_gate` does *not* increment, the recorder does
*not* see a pre_tool event, because the pipeline refused before
either fired.

In [26]:
before_count = limiter.count
before_events = len(recorder.events)

try:
    await dispatch(
        pipeline=stacked_pipeline,
        bus=stack_bus,
        tool_name="bash",
        arguments={"cmd": "ls"},
        execute=echo,
    )
except PolicyDenied as exc:
    print("policy denied:", exc.decision.layer, exc.decision.rule_id)

print("limiter count delta :", limiter.count - before_count)
print("recorder events delta :", len(recorder.events) - before_events)

policy denied: global global.denylist
limiter count delta : 0
recorder events delta : 0


There's no bus teardown step — the handlers stay subscribed. The final
tallies below reflect everything the stacked run observed.

In [27]:
# No bus.shutdown(): handlers persist. Final state of the stacked run:
print("limiter final count :", limiter.count)
print("recorder events     :", len(recorder.events))
print("pre_tool handlers   :", stack_bus.handler_count("agent:pre_tool"))

limiter final count : 3
recorder events     : 5
pre_tool handlers   : 2


## 10. Summary

**Two layers, one boundary.** ArcAgent runs `arctrust.policy`
first and `arcagent.core.module_bus` second on every tool
dispatch. Either layer can refuse, and they refuse with
different exception types so auditors can tell them apart.

**API recap:**

- **`ModuleBus`** — `subscribe`, `emit`, `handler_count`,
  `handler_count_by_module`. No module registry, no lifecycle:
  it is a pure subscribe/dispatch fabric.
- **Extensions** — plain objects (or module-level functions) that
  call `bus.subscribe(event, handler, priority=..., module_name=...)`.
  On disk they're `@hook` / `@capability` / `@background_task`
  decorated and bridged onto the bus by the `CapabilityLoader`.
- **`ModuleContext`** — frozen DI record (`bus`, `tool_registry`,
  `config`, `telemetry`, `workspace`, `llm_config`) handed to a
  `@capability` class's `setup(ctx)`.
- **`EventContext`** — `event`, `data` (snapshot copy),
  `agent_did`, `trace_id`, `veto(reason)`, `is_vetoed`,
  `veto_reason` (first-veto-wins).
- **`ToolVetoedError`** — raised when an `agent:pre_tool`
  handler vetoes. Subclass of `ToolError`.
- **`ModuleBusError`** — structured exception a *caller* raises for
  bus/extension lifecycle failures. Handler crashes/timeouts are
  isolated and do *not* surface as `ModuleBusError`.
- **`PolicyPipeline`, `PolicyContext`, `ToolCall`, `build_pipeline`,
  `PolicyDenied`** — re-exported through
  `arcagent.core.tool_policy`; the engine itself lives in
  `arctrust.policy`.

**Failure modes:**

| Trigger | Surface | Subclass of |
|---|---|---|
| Policy layer DENY | `PolicyDenied` | `ArcAgentError` |
| Bus handler `ctx.veto(...)` | `ToolVetoedError` | `ToolError` |
| Caller-raised lifecycle assertion | `ModuleBusError` | `ArcAgentError` |
| Handler exception or timeout | (logged, swallowed) | — |

**Next:** `04-module-bus-events.ipynb` zooms in on the event
catalogue itself — every event ArcAgent emits during a run,
their data shapes, and the capability lifecycle. That
notebook builds on the bus you saw here; this one was the
*shape* of the surface, the next one is the *vocabulary*.